# 04 — Evaluation, recovery and evidence gates

Purpose: independently evaluate one frozen AHR-only 9×9 player against the pinned GNU Go 3.8 level 1, and distinguish verified tactical learning from operational correctness. This notebook imports only `ahr-export` definitions; it does not run upstream training or generation cells.

Inputs: v5 tagged source definitions; the pinned GNU binary; a real trained checkpoint for strength runs; curriculum development rows for learning gates. Outputs: `reports/ahr_native_v5/arena_validation.json`, gate receipts, and separate development, confirmation and final reports. Every timeout, illegal emission or move cutoff is UNKNOWN, never a loss or UNSAT.

Expected runtime: recovery Run All about 1–3 minutes CPU; optional tactical/search gates 1–5 minutes GPU 1; 16/32-game development blocks about 0.5–2 hours; the 200-game final is permitted only after at least 50% observed wins in development and fresh confirmation. All arenas checkpoint every game boundary, 32 moves and at most 5 minutes. They retain the latest two valid resumable checkpoints plus the best. One writer owns each run.

Checkpoint location: `checkpoints/<eval_run>/`. Exact resume: fresh project kernel, Run All to load and verify definitions, then repeat `v5_run_gnu_evaluation(project, training_run, eval_run, split, games, checkpoint_path, device)` with unchanged arguments. The protected frozen copy survives training retention. `deadline_unix` pauses at a safe checkpoint without scoring an unfinished game. Source/protocol/checkpoint changes are rejected.

Final opening seeds belong to a separate reserved domain. The final lockbox permits one frozen request only; final evidence cannot select a new candidate. Confidence intervals account for paired colors from the same opening. No automatic promotion or claim of full-game optimality occurs.

In [ ]:
from pathlib import Path
import nbformat, json
project=Path.cwd().resolve()
while not (project/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if project.parent==project: raise RuntimeError('Run inside the AHR Go project')
    project=project.parent
namespace=globals()
shared=nbformat.read(project/'experiments/ahr_native_v5/00_shared.ipynb',4)
for cell in shared.cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
        exec(compile(cell.source,'v5 shared export','exec'),namespace)
v5_load_v4(project,namespace)
for name in ('01_model_and_search.ipynb','02_verified_curriculum.ipynb','03_learning.ipynb'):
    path=project/V5_ROOT/name
    if path.exists(): v5_import_exports(path,namespace)


## Frozen protocol and paired uncertainty

In [ ]:
from dataclasses import dataclass, asdict
from collections import Counter
import math, random, re, statistics, time, copy, hashlib, json, uuid
import numpy as np
import torch

V5_EVAL_SCHEMA = 'ahr-native-v5-gnu-evaluation-v1'
V5_EVAL_RESUME = '04: fresh project kernel; import only tagged 00–04 definitions; repeat v5_run_gnu_evaluation with unchanged arguments'
V5_EVAL_SEED_DOMAINS = {'development': 1100000000, 'confirmation': 1300000000, 'final': 1500000000, 'recovery_fixture': 1700000000}

@dataclass(frozen=True)
class V5GNUProtocol:
    split: str = 'development'
    games: int = 32
    seed: int = 1100000001
    size: int = 9
    komi: float = 7.5
    opening_plies: int = 2
    max_plies: int = 512
    ahr_budget_s: float = 1.0
    gnu_budget_s: float = 5.0
    gnu_command_timeout_s: float = 3.0
    level: int = 1
    alpha: float = .05
    def __post_init__(self):
        if self.split not in V5_EVAL_SEED_DOMAINS: raise ValueError('Unrecognized data split')
        if self.split == 'final' and self.games != 200: raise ValueError('Final benchmark is exactly 200 attempts')
        if self.split == 'development' and self.games not in (16,32,64): raise ValueError('Development blocks contain 16, 32 or 64 games')
        if self.split == 'confirmation' and self.games not in (32,64): raise ValueError('Fresh confirmation blocks contain 32 or 64 games')
        if type(self.games) is not int or self.games < 2 or self.games % 2: raise ValueError('Require complete black/white opening pairs')
        if self.size != 9 or self.komi != 7.5 or self.level != 1: raise ValueError('Pinned no-handicap 9x9/7.5/level1 protocol')
        if self.ahr_budget_s != 1.0 or self.gnu_budget_s != 5.0 or self.gnu_command_timeout_s != 3.0:
            raise ValueError('Pinned decision and exchange allowances')
        if not 0 <= self.opening_plies < self.max_plies or self.max_plies > 512: raise ValueError('Invalid chronology bounds')
        if not 0 < self.alpha < 1: raise ValueError('Invalid confidence parameter')

def v5_eval_path(project, split, name):
    if split not in V5_EVAL_SEED_DOMAINS or Path(name).name != name: raise ValueError('Unsafe split/report name')
    path = Path(project)/'reports/ahr_native_v5'/split/name
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

def v5_eval_schedule(protocol):
    domain = V5_EVAL_SEED_DOMAINS[protocol.split]
    rng = random.Random(protocol.seed)
    seeds = rng.sample(range(domain,domain+100000000),protocol.games//2)
    openings = []
    for seed in seeds:
        board = initial_state(protocol.size,protocol.komi)
        local = random.Random(seed); actions = []
        for _ in range(protocol.opening_plies):
            legal = [a for a in legal_actions(board) if a != board.pass_action]
            action = local.choice(legal); actions.append(action); board = play(board,action)
        openings.append({'seed':seed,'actions':actions,'state':to_dict(board)})
    jobs = [{'index':2*i+j,'block':i,'seed':s,'candidate_color':color}
            for i,s in enumerate(seeds) for j,color in enumerate((BLACK,WHITE))]
    return openings,jobs

def v5_eval_identity(project, protocol, players, candidate_receipt=None):
    if set(players) != {'candidate','reference'}: raise ValueError('Require native AHR and one reference')
    for player in players.values(): player.check_frozen()
    openings,jobs = v5_eval_schedule(protocol)
    return json.loads(arena_json({'schema':V5_EVAL_SCHEMA,'protocol':asdict(protocol),
        'rules':RULESET,'handicap':0,'sources':v5_source_identity(project,require_all=False),
        'openings':openings,'jobs':jobs,'players':{name:p.identity() for name,p in players.items()},
        'candidate_receipt':candidate_receipt,
        'gnugo_runtime':'fresh pinned process every call; complete chronology and state-derived seed; no hidden continuing engine state',
        'outcome_source':'independent project area scorer; no GNU final_score or suggested-move labels'}))

def v5_eval_statistics(identity, results):
    protocol = identity['protocol']; planned = protocol['games']
    complete = [r for r in results if r['status'] == 'COMPLETE']
    wins = sum(r['candidate_score'] == 1.0 for r in complete)
    losses = sum(r['candidate_score'] == 0.0 for r in complete)
    draws = sum(r['candidate_score'] == .5 for r in complete)
    unknown = len(results)-len(complete); pending = planned-len(results)
    grouped = {}
    for row in results: grouped.setdefault(row['job']['block'],[]).append(row)
    valid_pairs = [rows for rows in grouped.values() if len(rows)==2 and all(r['status']=='COMPLETE' for r in rows)]
    paired_values = [sum(r['candidate_score']==1.0 for r in pair)/2 for pair in valid_pairs]
    # Each independent opening contributes one bounded [0,1] paired win fraction.
    # No independence assumption between the two colors of one opening is made.
    interval = None
    if paired_values:
        mean = sum(paired_values)/len(paired_values)
        radius = math.sqrt(math.log(2/protocol['alpha'])/(2*len(paired_values)))
        interval = [max(0.,mean-radius),min(1.,mean+radius)]
    all_valid = len(complete)==planned and unknown==0 and pending==0
    margins = [r['black_score']*r['job']['candidate_color'] for r in complete]
    audits = [a for r in results for a in r['moves']]
    latency = {}
    for name,budget in [('candidate',protocol['ahr_budget_s']),('reference',protocol['gnu_budget_s'])]:
        times = [a['elapsed_s'] for a in audits if a['player']==name]
        latency[name] = {'calls':len(times),'mean_s':statistics.mean(times) if times else None,
                         'max_s':max(times) if times else None,'over_budget':sum(t>budget for t in times),
                         'allocated_budget_s':budget}
    return {'games_planned':planned,'games_recorded':len(results),'games_complete':len(complete),
        'wins':wins,'losses':losses,'draws':draws,'unknown':unknown,'pending':pending,
        'win_rate_completed':wins/len(complete) if complete else None,'wins_over_planned':wins/planned,
        'possible_win_rate_bounds':[wins/planned,(wins+unknown+pending)/planned],
        'mean_score_margin':statistics.mean(margins) if margins else None,
        'paired_blocks_complete':len(paired_values),'paired_confidence_interval':interval,
        'confidence_method':'two-sided Hoeffding interval over independent paired opening blocks; conditional on one frozen model and declared opening distribution',
        'confidence_level':1-protocol['alpha'],'all_games_valid':all_valid,
        'observed_half_wins':bool(all_valid and wins>=planned/2),
        'statistically_supported_above_half':bool(all_valid and interval and interval[0]>.5),
        'unknown_by_reason':dict(Counter(r['status'] for r in results if r['status']!='COMPLETE')),
        'latency_by_player':latency,'promotion_effect':'NONE','full_game_optimality':'NOT_ESTABLISHED'}


## Exact resumable chronology and independent CPU scoring

In [ ]:
def v5_eval_replay(identity, job, actions):
    board = initial_state(identity['protocol']['size'],identity['protocol']['komi'])
    opening = identity['openings'][job['block']]
    for action in opening['actions']: board = play(board,action)
    if to_dict(board)!=opening['state']: raise RuntimeError('Opening chronology changed')
    for action in actions: board = play(board,action)
    return board

def v5_eval_strict_load(run_dir):
    run_dir = Path(run_dir); manifest_path = run_dir/'manifest.json'
    if not manifest_path.exists():
        if list(run_dir.glob('step-*.pt')): raise RuntimeError('Missing evaluation manifest; refusing restart')
        return None
    try:
        manifest = v5_read_json(manifest_path); entries=manifest['entries']; states={}
        if not entries or len({e['file'] for e in entries})!=len(entries): raise ValueError('Bad checkpoint inventory')
        for entry in entries:
            if Path(entry['file']).name!=entry['file']: raise ValueError('Unsafe checkpoint path')
            state=_checkpoint_entry_state(run_dir,entry)
            if state is None: raise ValueError('Invalid committed evaluation checkpoint: '+entry['file'])
            states[entry['file']]=state
        if manifest['latest'] not in states or manifest['best'] not in states: raise ValueError('Missing latest/best')
        latest=states[manifest['latest']]
        if latest['global_step']!=max(s['global_step'] for s in states.values()): raise ValueError('Rewound latest cursor')
        return latest
    except (KeyError,ValueError,TypeError,OSError) as error:
        raise RuntimeError('Invalid evaluation manifest; refusing silent recovery') from error

def v5_eval_validate_snapshot(snapshot,identity,verify_history=True):
    if REQUIRED_CHECKPOINT_KEYS-set(snapshot): raise RuntimeError('Incomplete evaluation resume payload')
    if snapshot.get('schema')!=V5_EVAL_SCHEMA or snapshot['config']!=identity or snapshot.get('identity_sha256')!=v5_digest(identity):
        raise RuntimeError('Frozen model, source, split or protocol changed')
    if any(snapshot[k] is not None for k in ('optimizer','scheduler','scaler','replay')) or snapshot['optimizer_steps']!=0:
        raise RuntimeError('Evaluation contains unexpected training state')
    if snapshot['distributed']!={'world_size':1}: raise RuntimeError('Unexpected evaluation distributed state')
    cursor=snapshot['sampler']['next_game'];jobs=identity['jobs']
    if type(cursor) is not int or not 0<=cursor<=len(jobs) or len(snapshot['results'])!=cursor:
        raise RuntimeError('Evaluation sampler/result cursor mismatch')
    if snapshot['complete']!=(cursor==len(jobs)): raise RuntimeError('Invalid completion marker')
    if type(snapshot['global_step']) is not int or snapshot['global_step']<0: raise RuntimeError('Invalid event cursor')
    for name,weights in snapshot['model'].items():
        if name not in identity['players'] or arena_tensor_digest(weights)!=identity['players'][name]['weights_sha256']:
            raise RuntimeError('Frozen model changed')
    if set(snapshot['model'])!=set(identity['players']) or set(snapshot['runtime'])!=set(identity['players']):
        raise RuntimeError('Model/runtime inventory mismatch')
    for i,row in enumerate(snapshot['results'] if verify_history else []):
        if row['job']!=jobs[i]: raise RuntimeError('Paired result ordering changed')
        board=v5_eval_replay(identity,row['job'],row['actions'])
        if to_dict(board)!=row['final_state']: raise RuntimeError('Result chronology differs from saved board')
        if row['status']=='COMPLETE':
            score=area_score(board);outcome=.5 if score==0 else float(score*row['job']['candidate_color']>0)
            if score!=arena_reference_score(board): raise RuntimeError('Independent final area scorers disagree')
            if not board.terminal or row['black_score']!=score or row['candidate_score']!=outcome:
                raise RuntimeError('Terminal result failed independent scoring')
        elif not row['status'].startswith('UNKNOWN_') or row['black_score'] is not None or row['candidate_score'] is not None:
            raise RuntimeError('UNKNOWN must not be scored as a loss')
    active=snapshot['active']
    if active is not None:
        if cursor>=len(jobs) or active['job']!=jobs[cursor]: raise RuntimeError('Active game cursor changed')
        if len(active['moves'])!=len(active['actions']): raise RuntimeError('Active move audit mismatch')
        if verify_history and to_dict(v5_eval_replay(identity,active['job'],active['actions']))!=active['state']:
            raise RuntimeError('Active game chronology differs from saved board')
    return True

def v5_eval_commit(run_dir,snapshot,players,first=False):
    for player in players.values(): player.check_frozen()
    if first: snapshot['model']={name:p.weights() for name,p in players.items()}
    snapshot['runtime']={name:p.runtime() for name,p in players.items()}
    snapshot['rng']=capture_rng()
    snapshot['metrics']={'recorded_games':len(snapshot['results']),
                         'complete_games':sum(r['status']=='COMPLETE' for r in snapshot['results'])}
    v5_eval_validate_snapshot(snapshot,snapshot['config'],verify_history=False)
    return _save_checkpoint_unlocked(run_dir,snapshot,best=first or snapshot['complete'],resume_cell=V5_EVAL_RESUME)

def v5_eval_finish(snapshot,status):
    active=snapshot['active'];board=from_dict(active['state'])
    score=area_score(board) if status=='COMPLETE' else None
    outcome=(.5 if score==0 else float(score*active['job']['candidate_color']>0)) if score is not None else None
    snapshot['results'].append({'job':active['job'],'actions':active['actions'],'moves':active['moves'],
        'final_state':active['state'],'status':status,'black_score':score,'candidate_score':outcome})
    snapshot['sampler']['next_game']+=1;snapshot['active']=None
    snapshot['complete']=snapshot['sampler']['next_game']==len(snapshot['config']['jobs'])

def v5_eval_run(project,eval_run,identity,players,stop_after_events=None,deadline_unix=None):
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]{0,100}',eval_run): raise ValueError('Simple eval run ID required')
    if stop_after_events is not None and (type(stop_after_events) is not int or stop_after_events<1): raise ValueError('Positive event cutoff required')
    run_dir=Path(project)/'checkpoints'/eval_run
    if identity['protocol']['split']!='recovery_fixture':
        validation=v5_read_json(v5_report_path(project,'arena_validation.json'))
        if validation.get('exact_resume') is not True or validation.get('sources')!=identity['sources']:
            raise RuntimeError('Current exact-resume validation is required before long evaluation')
    with checkpoint_lock(run_dir):
        snapshot=v5_eval_strict_load(run_dir)
        if snapshot is None:
            if (run_dir/'identity.json').exists() and v5_read_json(run_dir/'identity.json')!=identity:
                raise RuntimeError('Frozen evaluation request changed')
            v5_atomic_json(run_dir/'identity.json',identity);v5_seed(identity['protocol']['seed'])
            snapshot={'schema':V5_EVAL_SCHEMA,'identity_sha256':v5_digest(identity),'config':identity,
                'model':{},'runtime':{},'initial_runtime':{name:p.runtime() for name,p in players.items()},
                'optimizer':None,'scheduler':None,'scaler':None,'rng':None,'sampler':{'next_game':0},
                'global_step':0,'optimizer_steps':0,'metrics':{},'replay':None,'distributed':{'world_size':1},
                'results':[],'active':None,'complete':False}
            v5_eval_commit(run_dir,snapshot,players,first=True)
        else:
            v5_eval_validate_snapshot(snapshot,identity)
            if not (run_dir/'identity.json').is_file() or v5_read_json(run_dir/'identity.json')!=identity:
                raise RuntimeError('Committed evaluation identity missing/changed')
            for name,player in players.items(): player.restore(snapshot['model'][name],snapshot['runtime'][name])
            restore_rng(snapshot['rng'])
        events=0;last_commit=time.monotonic();moves_since_commit=0
        while not snapshot['complete']:
            if deadline_unix is not None and time.time() >= deadline_unix:
                manifest=v5_read_json(run_dir/'manifest.json')
                if snapshot['global_step'] > max(e['global_step'] for e in manifest['entries']):
                    v5_eval_commit(run_dir,snapshot,players)
                break
            boundary=False
            if snapshot['active'] is None:
                job=identity['jobs'][snapshot['sampler']['next_game']];v5_seed(job['seed'])
                for name,player in players.items():
                    if player.set_runtime_state: player.set_runtime_state(copy.deepcopy(snapshot['initial_runtime'][name]))
                snapshot['active']={'job':job,'state':copy.deepcopy(identity['openings'][job['block']]['state']),
                                    'actions':[],'moves':[]}
                boundary=True
            else:
                active=snapshot['active'];board=from_dict(active['state']);protocol=identity['protocol']
                if board.terminal: v5_eval_finish(snapshot,'COMPLETE');boundary=True
                elif board.move_number>=protocol['max_plies']: v5_eval_finish(snapshot,'UNKNOWN_MOVE_CAP');boundary=True
                else:
                    name='candidate' if board.to_play==active['job']['candidate_color'] else 'reference'
                    player=players[name];budget=protocol['ahr_budget_s'] if name=='candidate' else protocol['gnu_budget_s']
                    device=next(player.model.parameters()).device if player.model is not None else None
                    if device is not None and device.type=='cuda': torch.cuda.synchronize(device)
                    started=time.perf_counter();prepared_at=started;failure=None;successor=None
                    try:
                        if player.prepare_position:
                            player.prepare_position(board,copy.deepcopy(identity['openings'][active['job']['block']]['actions']),copy.deepcopy(active['actions']))
                        prepared_at=time.perf_counter()
                        with torch.inference_mode(): result=player.decide(board,budget)
                        if device is not None and device.type=='cuda': torch.cuda.synchronize(device)
                    except Exception as error:
                        result={'action':None,'exception':type(error).__name__+': '+str(error)[:250]};failure='UNKNOWN_PLAYER_ERROR'
                    elapsed=time.perf_counter()-started
                    if not isinstance(result,dict): result={'action':None};failure='UNKNOWN_PLAYER_ERROR'
                    action=result.get('action')
                    if elapsed>budget or result.get('deadline_missed',False): failure='UNKNOWN_DEADLINE'
                    try:
                        successor=play(board,action)
                        if not verify_go_emission(board,action,successor): raise RuntimeError('Independent verifier rejected transition')
                        legal=True
                    except (IllegalMove,TypeError,ValueError):
                        legal=False;successor=None
                        if failure is None: failure='UNKNOWN_ILLEGAL_MOVE'
                    audit={'player':name,'action':action if type(action) is int else None,'legal':legal,'elapsed_s':elapsed,
                        'allocated_budget_s':budget,'preparation_s':prepared_at-started,
                        'decision_and_exchange_s':max(0.,elapsed-(prepared_at-started)),
                        'deadline_missed':elapsed>budget or bool(result.get('deadline_missed',False)),
                        'failure':failure,'strategic_status':'UNKNOWN',
                        'counters':{k:result[k] for k in ('completed_depth','nodes','model_calls','root_value','reference_seed')
                                    if k in result and type(result[k]) in (int,float,str,bool)},
                        'rng_probe':result.get('rng_probe'),'exception':result.get('exception')}
                    active['moves'].append(audit)
                    if failure: v5_eval_finish(snapshot,failure);boundary=True
                    else:
                        active['actions'].append(action);active['state']=to_dict(successor)
                    moves_since_commit+=1
            snapshot['global_step']+=1;events+=1
            stopping=stop_after_events is not None and events>=stop_after_events
            if boundary or stopping or moves_since_commit>=32 or time.monotonic()-last_commit>=300:
                v5_eval_commit(run_dir,snapshot,players);last_commit=time.monotonic();moves_since_commit=0
                progress=v5_eval_statistics(identity,snapshot['results'])
                v5_atomic_json(run_dir/'progress.json',{'status':'COMPLETE' if snapshot['complete'] else 'IN_PROGRESS',
                    'global_step':snapshot['global_step'],'active_move':len(snapshot['active']['actions']) if snapshot['active'] else None,
                    **progress})
            if stopping: break
        for player in players.values(): player.check_frozen()
        return snapshot,v5_eval_statistics(identity,snapshot['results'])


## Protected checkpoint selection and independent failure diagnostics

In [ ]:
def v5_eval_failures(identity,results):
    rows=[]
    for result in results:
        color=result['job']['candidate_color'];board=v5_eval_replay(identity,result['job'],[])
        first_loss=None;total_lost=0;first_atari_ignored=None;depths=[]
        for i,action in enumerate(result['actions']):
            if board.to_play==color:
                endangered=[sorted(g) for g,l in groups_and_liberties(board.board,board.size)
                            if board.board[next(iter(g))]==color and len(l)==1]
                child=play(board,action)
                if endangered and first_atari_ignored is None:
                    unresolved=[g for g in endangered if any(child.board[p]==color for p in g)
                                and len(group_and_liberties(child.board,child.size,next(p for p in g if child.board[p]==color))[1])==1]
                    if unresolved: first_atari_ignored={'move_number':board.move_number,'action':action,'groups':unresolved,
                        'interpretation':'Group remained in atari after this move; this is a diagnostic flag, not proof that defense was best'}
            else: child=play(board,action)
            removed=[p for p in range(board.size**2) if board.board[p]==color and child.board[p]!=color]
            if removed:
                total_lost+=len(removed)
                if first_loss is None:
                    seen=set();groups=[]
                    for p in removed:
                        if p not in seen:
                            group,liberties=group_and_liberties(board.board,board.size,p);seen.update(group)
                            groups.append({'stones':sorted(group),'liberties_before_capture':sorted(liberties)})
                    first_loss={'move_number':board.move_number,'capturing_action':action,'groups':groups,'stones_lost':len(removed)}
            board=child
        for audit in result['moves']:
            if audit['player']=='candidate' and 'completed_depth' in audit['counters']:
                depths.append(audit['counters']['completed_depth'])
        rows.append({'game_index':result['job']['index'],'candidate_color':color,'status':result['status'],
            'won':result['candidate_score']==1 if result['status']=='COMPLETE' else None,
            'first_lost_group':first_loss,'stones_captured_over_game':total_lost,
            'surviving_candidate_stones':sum(v==color for v in board.board),
            'first_unresolved_atari':first_atari_ignored,'completed_depth_counts':dict(Counter(depths))})
    return {'games':rows,'games_with_zero_surviving_stones':sum(r['surviving_candidate_stones']==0 for r in rows),
        'scope':'Descriptive replay diagnostics; early capture and unresolved atari do not establish the causal losing move'}

def v5_eval_pin_model(project,training_run,eval_run,checkpoint_path,device):
    run_dir=Path(project)/'checkpoints'/eval_run;pin=run_dir/'frozen_model.pt';receipt_path=run_dir/'frozen_model.json'
    with checkpoint_lock(Path(project)/'checkpoints'/('pin-'+eval_run)):
        if pin.exists()!=receipt_path.exists(): raise RuntimeError('Incomplete frozen model pin; inspect rather than restart')
        if pin.exists():
            declared=v5_read_json(receipt_path)
            if declared['training_run']!=training_run or file_sha256(pin)!=declared['checkpoint_sha256']:
                raise RuntimeError('Protected model pin identity changed')
            if checkpoint_path is not None:
                supplied=Path(checkpoint_path)
                if not supplied.is_absolute(): supplied=Path(project)/supplied
                supplied=supplied.resolve()
                if not supplied.is_relative_to(Path(project).resolve()): raise RuntimeError('Pinned checkpoint must belong to this workspace')
                allowed_missing=(Path(project)/declared['original_checkpoint_path']).resolve()
                if not supplied.exists() and supplied!=allowed_missing: raise RuntimeError('Requested missing checkpoint is not the original protected source')
                if supplied.exists() and file_sha256(supplied)!=declared['checkpoint_sha256']:
                    raise RuntimeError('Requested checkpoint differs from frozen model')
            model,receipt=v5_load_model(project,training_run,device=device,checkpoint_path=str(pin.relative_to(project)))
        else:
            model,receipt=v5_load_model(project,training_run,device=device,checkpoint_path=checkpoint_path)
            source=Path(receipt['checkpoint_path'])
            if not source.is_absolute(): source=Path(project)/source
            run_dir.mkdir(parents=True,exist_ok=True)
            atomic_write_bytes(pin,source.read_bytes())
            if file_sha256(pin)!=receipt['checkpoint_sha256']: raise RuntimeError('Protected model copy hash mismatch')
            v5_atomic_json(receipt_path,{'training_run':training_run,'checkpoint_sha256':receipt['checkpoint_sha256'],
                'original_checkpoint_path':receipt['checkpoint_path'],'protected_checkpoint_path':str(pin.relative_to(project))})
        if receipt.get('purpose') not in ('go_training','native_go_training') or receipt.get('optimizer_steps',0)<1:
            raise RuntimeError('Strength evaluation requires actual AHR training, not synthetic recovery fixtures')
        frozen={k:receipt.get(k) for k in ('checkpoint_sha256','global_step','optimizer_steps','games_finished','config','model_config','source_identity','purpose')}
        frozen.update(training_run=training_run,checkpoint_path=str(pin.relative_to(project)))
        return model,frozen

def v5_run_gnu_evaluation(project,training_run,eval_run,split='development',games=32,checkpoint_path=None,
                         device='cuda:1',stop_after_events=None,deadline_unix=None):
    project=Path(project).resolve()
    if split not in ('development','confirmation','final'): raise ValueError('Public arena accepts development, confirmation or final only')
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]{0,100}',eval_run): raise ValueError('Simple evaluation run ID required')
    model,receipt=v5_eval_pin_model(project,training_run,eval_run,checkpoint_path,device)
    if split=='final': v5_require_final_ready(project,training_run,receipt['checkpoint_path'])
    model.eval();model.requires_grad_(False)
    search=V5SearchConfig(time_budget_s=1.0)
    # Warm model and search before any measured call, including after a resumed VM/kernel.
    with torch.inference_mode():
        warm_state=initial_state(9,7.5)
        for _ in range(2): v5_choose_move(model,warm_state,search)
    if next(model.parameters()).device.type=='cuda': torch.cuda.synchronize(next(model.parameters()).device)
    candidate=FrozenArenaPlayer('candidate',lambda state,budget:v5_choose_move(model,state,search),
        {'search':asdict(search),'ahr_budget_s':1.0,'architecture':'native Queen/Worker/Scout with coherent root rounds','jepa':False},
        {'kind':'ahr_native_v5_checkpoint',**receipt},model=model)
    domain=V5_EVAL_SEED_DOMAINS[split]
    # Opening seeds depend on eval_run and split, never on observed results or training RNG.
    seed=domain+int(hashlib.sha256((split+':'+eval_run).encode()).hexdigest()[:12],16)%100000000
    protocol=V5GNUProtocol(split=split,games=games,seed=seed)
    reference=arena_gnugo_player(project,'reference',level=1,seed=seed,command_timeout_s=3.0)
    reference.config['allocated_exchange_budget_s']=5.0
    players={'candidate':candidate,'reference':reference}
    identity=v5_eval_identity(project,protocol,players,receipt)
    request_path=v5_eval_path(project,split,eval_run+'-request.json')
    with checkpoint_lock(project/'checkpoints'/('request-'+eval_run)):
        if request_path.exists() and v5_read_json(request_path)!=identity: raise RuntimeError('Frozen request changed')
        if not request_path.exists(): v5_atomic_json(request_path,identity)
    if split=='final': v5_commit_final_lockbox(project,eval_run,identity)
    snapshot,stats=v5_eval_run(project,eval_run,identity,players,stop_after_events=stop_after_events,deadline_unix=deadline_unix)
    report_path=v5_eval_path(project,split,eval_run+'-results.json')
    report={'schema':V5_EVAL_SCHEMA,'status':'COMPLETE' if snapshot['complete'] else 'IN_PROGRESS',
        'split':split,'eval_run':eval_run,'training_run':training_run,'candidate':receipt,'identity_sha256':v5_digest(identity),
        'statistics':stats,'failure_analysis':v5_eval_failures(identity,snapshot['results']),
        'results':snapshot['results'],'final_data_used_for_training':False,
        'selected_player_changed':False,'unknown_semantics':'UNKNOWN is unscored; denominator and possible bounds include all planned attempts'}
    v5_atomic_json(report_path,report)
    returned={k:receipt[k] for k in ('checkpoint_path','checkpoint_sha256','optimizer_steps','games_finished')}
    returned.update(schema=V5_EVAL_SCHEMA,status=report['status'],split=split,eval_run=eval_run,training_run=training_run,
                    report_path=str(report_path.relative_to(project)),**stats)
    v5_atomic_json(v5_eval_path(project,split,eval_run+'-receipt.json'),returned)
    return returned


## Recovery fixtures and honest UNKNOWN semantics

In [ ]:
def v5_eval_fixture_players(mode='normal'):
    def decide(state,budget):
        probe=[random.random(),float(np.random.random()),float(torch.rand(()))]
        if mode=='illegal': action=-7
        elif mode=='cutoff': action=next(a for a in legal_actions(state) if a!=state.pass_action)
        elif mode=='deadline': return {'action':state.pass_action,'deadline_missed':True,'rng_probe':probe}
        elif state.move_number>=4: action=state.pass_action
        else:
            choices=[a for a in legal_actions(state) if a!=state.pass_action]
            action=choices[int(probe[0]*len(choices))%len(choices)]
        return {'action':action,'rng_probe':probe,'completed_depth':0,'nodes':0,'deadline_missed':False}
    return {name:FrozenArenaPlayer(name,decide,{'fixture':mode},{'kind':'fixture','purpose':'recovery_only'})
            for name in ('candidate','reference')}

def v5_eval_semantic(snapshot):
    rows=[]
    for row in snapshot['results']:
        clean=copy.deepcopy(row)
        for audit in clean['moves']:
            for key in ('elapsed_s','preparation_s','decision_and_exchange_s'): audit.pop(key,None)
        rows.append(clean)
    return {'results':rows,'sampler':snapshot['sampler'],'global_step':snapshot['global_step'],'complete':snapshot['complete']}

def v5_eval_fixture_subprocess(project,eval_run,identity,stop_after_events=None):
    import subprocess
    project=Path(project).resolve()
    request=project/'reports/ahr_native_v5/recovery_fixture'/(eval_run+'-fixture-request.json')
    v5_atomic_json(request,identity)
    program = """import json,sys
from pathlib import Path
import nbformat
project=Path(sys.argv[1]);request=Path(sys.argv[2]);run=sys.argv[3]
namespace={'__name__':'__main__'}
book=nbformat.read(project/'experiments/ahr_native_v5/00_shared.ipynb',4)
for cell in book.cells:
 if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'00:export','exec'),namespace)
namespace['v5_load_v4'](project,namespace)
namespace['v5_import_exports'](project/'experiments/ahr_native_v5/04_evaluation_and_gates.ipynb',namespace)
identity=json.loads(request.read_text());cutoff=None if sys.argv[4]=='None' else int(sys.argv[4])
snapshot,stats=namespace['v5_eval_run'](project,run,identity,namespace['v5_eval_fixture_players'](),stop_after_events=cutoff)
print('V5_ARENA_FIXTURE='+namespace['v5_digest'](namespace['v5_eval_semantic'](snapshot)))
"""
    process=subprocess.run([str(project/'.venv/bin/python'),'-c',program,str(project),str(request),eval_run,str(stop_after_events)],
                           cwd=project,capture_output=True,text=True,timeout=90,check=True)
    markers=[line.split('=',1)[1] for line in process.stdout.splitlines() if line.startswith('V5_ARENA_FIXTURE=')]
    if len(markers)!=1:raise RuntimeError('Fresh arena recovery worker did not emit exact-state digest')
    return markers[0]

def v5_validate_arena_resume(project,device='cuda:1'):
    project=Path(project).resolve();suffix=uuid.uuid4().hex[:10]
    protocol=V5GNUProtocol(split='recovery_fixture',games=2,seed=1700000017,opening_plies=0,max_plies=16)
    players=v5_eval_fixture_players();identity=v5_eval_identity(project,protocol,players)
    whole,_=v5_eval_run(project,'ahr-v5-arena-whole-'+suffix,identity,players)
    split_run='ahr-v5-arena-resume-'+suffix
    partial,_=v5_eval_run(project,split_run,identity,v5_eval_fixture_players(),stop_after_events=3)
    if partial['active'] is None or len(partial['active']['actions'])!=2: raise AssertionError('Did not interrupt midgame')
    # Deliberately perturb all RNGs before restoration.
    random.random();np.random.random();torch.rand(())
    fresh_digest=v5_eval_fixture_subprocess(project,split_run,identity)
    resumed,_=v5_eval_run(project,split_run,identity,v5_eval_fixture_players())
    if fresh_digest!=v5_digest(v5_eval_semantic(whole)): raise AssertionError('Fresh process resumed chronology/RNG differs')
    if v5_eval_semantic(whole)!=v5_eval_semantic(resumed): raise AssertionError('Exact resumed chronology/RNG differs')
    retention=v5_read_json(project/'checkpoints'/split_run/'manifest.json')
    if len(retention['entries'])<2 or len(retention['entries'])>3: raise AssertionError('latest2+best retention failed')
    rejected=[]
    bad=copy.deepcopy(identity);bad['protocol']['komi']=6.5
    try: v5_eval_run(project,split_run,bad,v5_eval_fixture_players())
    except RuntimeError: rejected.append('changed_identity')
    else: raise AssertionError('Changed protocol accepted')
    for mode,expected in [('illegal','UNKNOWN_ILLEGAL_MOVE'),('deadline','UNKNOWN_DEADLINE'),('cutoff','UNKNOWN_MOVE_CAP')]:
        test_protocol=V5GNUProtocol(split='recovery_fixture',games=2,seed=1700000027,opening_plies=0,max_plies=4)
        test_players=v5_eval_fixture_players(mode);test_identity=v5_eval_identity(project,test_protocol,test_players)
        unknown,_=v5_eval_run(project,'ahr-v5-arena-'+mode+'-'+suffix,test_identity,test_players)
        if any(r['status']!=expected or r['candidate_score'] is not None for r in unknown['results']):
            raise AssertionError('UNKNOWN scoring regression: '+mode)
    # Actual pinned GNU process, full-history replay, independent emitted-move legality,
    # deterministic repeat of the same restored chronology and state-derived seed.
    binary=gnugo10_binary_identity(project)
    opening=initial_state(9,7.5);actions=[40,30]
    for action in actions: opening=play(opening,action)
    implementation=GNU10ReplayPlayer(project,level=1,seed=1700000031,command_timeout_s=3.0)
    implementation.prepare_position(opening,[],actions);one=implementation(opening,5.0)
    implementation.prepare_position(opening,[],actions);two=implementation(opening,5.0)
    if one['action']!=two['action'] or one['reference_seed']!=two['reference_seed'] or not one['legality_verified']:
        raise AssertionError('GNU replay recovery changed move/seed')
    if GNUGO10_RULE_FLAGS!=['--chinese-rules','--positional-superko','--forbid-suicide','--capture-all-dead','--never-resign']:
        raise AssertionError('GNU rule flags changed')
    # Check corruption rejection without modifying a valid run.
    corrupt_dir=project/'checkpoints'/('ahr-v5-arena-corrupt-'+suffix)
    with checkpoint_lock(corrupt_dir):
        corrupt=copy.deepcopy(partial);v5_eval_commit(corrupt_dir,corrupt,v5_eval_fixture_players(),first=True)
    manifest=v5_read_json(corrupt_dir/'manifest.json');entry=manifest['entries'][0]
    target=corrupt_dir/entry['file'];data=bytearray(target.read_bytes());data[-1]^=1;atomic_write_bytes(target,data)
    try: v5_eval_strict_load(corrupt_dir)
    except RuntimeError: rejected.append('corrupt_payload')
    else: raise AssertionError('Corrupted checkpoint accepted')
    receipt={'schema':V5_EVAL_SCHEMA,'status':'PASS','exact_resume':True,
        'sources':v5_source_identity(project,require_all=False),'checks':{'midgame_rng_chronology':True,'fresh_process_resume':True,'unknown_semantics':True,
        'latest2_best_retention':True,'immutable_request':True,'corrupt_payload_rejected':True,
        'gnu_pinned_binary':True,'gnu_chronological_repeat':True,'rules_and_color_pairs':True},
        'gnu_identity':binary,'gnu_smoke_elapsed_s':[one['elapsed_s'],two['elapsed_s']],
        'fixture_scope':'recovery wiring only; no playing-strength claim; actual V5 model validated separately in 01/02',
        'fixture_runs':['ahr-v5-arena-whole-'+suffix,split_run],
        'device_for_strength_runs':device,'resume_cell':V5_EVAL_RESUME}
    v5_atomic_json(v5_report_path(project,'arena_validation.json'),receipt)
    return receipt


## Final readiness, single lockbox and real GNU midgame recovery

In [ ]:
def v5_commit_final_lockbox(project,eval_run,identity):
    project=Path(project).resolve()
    lockbox_path=project/'config/ahr_native_v5_final_lockbox.json'
    lockbox={'eval_run':eval_run,'identity_sha256':v5_digest(identity),'identity':identity,
        'access':'final-only; excluded from trainer and development feedback','promotion_effect':'NONE'}
    with checkpoint_lock(project/'checkpoints'/'ahr-native-v5-final-lockbox'):
        if lockbox_path.exists() and v5_read_json(lockbox_path)!=lockbox:
            raise RuntimeError('One final candidate/protocol already frozen; changing candidate after final access is forbidden')
        if not lockbox_path.exists(): v5_atomic_json(lockbox_path,lockbox)
    return lockbox

def v5_verified_ready_blocks(project,split,training_run,checkpoint_sha256):
    if split not in ('development','confirmation'): raise ValueError('Readiness uses development and fresh confirmation only')
    project=Path(project).resolve();sources=v5_source_identity(project);valid=[]
    directory=project/'reports/ahr_native_v5'/split
    for receipt_path in sorted(directory.glob('*-receipt.json')) if directory.exists() else []:
        receipt=v5_read_json(receipt_path)
        if receipt.get('status')!='COMPLETE' or receipt.get('training_run')!=training_run:
            continue
        if receipt.get('checkpoint_sha256')!=checkpoint_sha256: continue
        report_path=project/receipt['report_path'];report=v5_read_json(report_path)
        request=v5_read_json(directory/(receipt['eval_run']+'-request.json'))
        if request['sources']!=sources or report['identity_sha256']!=v5_digest(request):
            raise RuntimeError('Readiness evidence sources/request changed')
        if request['protocol']['split']!=split or report['candidate']['checkpoint_sha256']!=checkpoint_sha256:
            raise RuntimeError('Readiness candidate/split mismatch')
        # Recompute independent chronology/scoring, rather than trust a summary receipt.
        for index,row in enumerate(report['results']):
            if row['job']!=request['jobs'][index]: raise RuntimeError('Readiness ordering changed')
            board=v5_eval_replay(request,row['job'],row['actions'])
            if to_dict(board)!=row['final_state']: raise RuntimeError('Readiness board chronology changed')
            if row['status']=='COMPLETE':
                score=area_score(board)
                expected=.5 if score==0 else float(score*row['job']['candidate_color']>0)
                if not board.terminal or score!=arena_reference_score(board) or row['black_score']!=score or row['candidate_score']!=expected:
                    raise RuntimeError('Readiness terminal score failed independent verification')
            elif not row['status'].startswith('UNKNOWN_') or row['candidate_score'] is not None:
                raise RuntimeError('Readiness UNKNOWN scoring invalid')
        stats=v5_eval_statistics(request,report['results'])
        if stats['all_games_valid'] and stats['wins_over_planned']>=.5:
            valid.append({'eval_run':receipt['eval_run'],'checkpoint_sha256':checkpoint_sha256,
                          'training_run':training_run,'seed':request['protocol']['seed'],
                          'games':stats['games_planned'],'wins':stats['wins'],
                          'observed_win_rate':stats['wins_over_planned'],'split':split})
    return valid

def v5_require_final_ready(project,training_run,checkpoint_path):
    project=Path(project).resolve()
    if checkpoint_path is None: raise RuntimeError('Final requires the exact protected candidate from development and confirmation')
    candidate=Path(checkpoint_path)
    if not candidate.is_absolute(): candidate=project/candidate
    candidate=candidate.resolve()
    if not candidate.is_relative_to(project) or not candidate.is_file(): raise RuntimeError('Final candidate checkpoint unavailable')
    candidate_sha=file_sha256(candidate)
    development=v5_verified_ready_blocks(project,'development',training_run,candidate_sha)
    confirmation=v5_verified_ready_blocks(project,'confirmation',training_run,candidate_sha)
    if not development or not confirmation:
        receipt={'status':'NOT_READY','checkpoint_sha256':candidate_sha,'training_run':training_run,
                 'development_qualifying_blocks':len(development),'confirmation_qualifying_blocks':len(confirmation),
                 'final_games_started':False,'requirement':'same frozen candidate: >=50% wins over all planned development and fresh confirmation games, zero UNKNOWN'}
        v5_atomic_json(v5_report_path(project,'final_readiness.json'),receipt)
        raise RuntimeError('Final NOT_READY: development and fresh confirmation have not both established observed >=50% wins for this exact candidate')
    if any(a['seed']==b['seed'] for a in development for b in confirmation):
        raise RuntimeError('Development and confirmation seeds must differ')
    receipt={'status':'READY','checkpoint_sha256':candidate_sha,'training_run':training_run,
             'development':development,'confirmation':confirmation,'source_identity':v5_source_identity(project),
             'observed_only':True,'true_win_probability_above_half':'NOT_ESTABLISHED'}
    v5_atomic_json(v5_report_path(project,'final_readiness.json'),receipt)
    return receipt

def v5_real_gnu_fixture_players(project):
    # Actual GNU decision before and after a resume, with deliberately short games.
    # The pass-after-one-call wrapper is a recovery fixture, never a strength opponent.
    project=Path(project).resolve();runtime={'calls':0}
    implementation=GNU10ReplayPlayer(project,level=1,seed=1700000041,command_timeout_s=3.0)
    def candidate(state,budget):
        probe=[random.random(),float(np.random.random()),float(torch.rand(()))]
        choices=[a for a in legal_actions(state) if a!=state.pass_action]
        action=state.pass_action if state.move_number>=4 else choices[int(probe[0]*len(choices))%len(choices)]
        return {'action':action,'rng_probe':probe,'deadline_missed':False}
    def reference(state,budget):
        if runtime['calls']:
            return {'action':state.pass_action,'deadline_missed':False,'reference_seed':-1}
        runtime['calls']+=1
        return implementation(state,budget)
    def restore(saved): runtime.clear();runtime.update(copy.deepcopy(saved))
    candidate_player=FrozenArenaPlayer('candidate',candidate,{'fixture':'random then pass'},{'kind':'fixture','purpose':'recovery_only'})
    reference_player=FrozenArenaPlayer('reference',reference,{'fixture':'actual GNU first reply then pass','binary':gnugo10_binary_identity(project)},
        {'kind':'fixture','purpose':'recovery_only'},get_runtime_state=lambda:copy.deepcopy(runtime),set_runtime_state=restore,
        prepare_position=implementation.prepare_position)
    return {'candidate':candidate_player,'reference':reference_player}

def v5_real_gnu_fixture_subprocess(project,eval_run,identity):
    import subprocess
    project=Path(project).resolve();request=v5_eval_path(project,'recovery_fixture',eval_run+'-actual-gnu-request.json')
    v5_atomic_json(request,identity)
    program="""import json,sys
from pathlib import Path
import nbformat
project=Path(sys.argv[1]);namespace={'__name__':'__main__'}
book=nbformat.read(project/'experiments/ahr_native_v5/00_shared.ipynb',4)
for cell in book.cells:
 if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'v5 shared export','exec'),namespace)
namespace['v5_load_v4'](project,namespace)
namespace['v5_import_exports'](project/'experiments/ahr_native_v5/04_evaluation_and_gates.ipynb',namespace)
identity=json.loads(Path(sys.argv[2]).read_text())
snapshot,stats=namespace['v5_eval_run'](project,sys.argv[3],identity,namespace['v5_real_gnu_fixture_players'](project))
print('V5_ACTUAL_GNU_RECOVERY='+namespace['v5_digest'](namespace['v5_eval_semantic'](snapshot)))
"""
    process=subprocess.run([str(project/'.venv/bin/python'),'-c',program,str(project),str(request),eval_run],cwd=project,
        capture_output=True,text=True,timeout=120,check=True)
    markers=[line.split('=',1)[1] for line in process.stdout.splitlines() if line.startswith('V5_ACTUAL_GNU_RECOVERY=')]
    if len(markers)!=1: raise RuntimeError('Fresh-process GNU recovery did not emit chronology digest')
    return markers[0]

v5_validate_arena_resume_base=v5_validate_arena_resume

def v5_validate_arena_resume(project,device='cuda:1'):
    project=Path(project).resolve();base=v5_validate_arena_resume_base(project,device=device)
    suffix=uuid.uuid4().hex[:10]
    protocol=V5GNUProtocol(split='recovery_fixture',games=2,seed=1700000043,opening_plies=0,max_plies=16)
    players=v5_real_gnu_fixture_players(project);identity=v5_eval_identity(project,protocol,players)
    whole,_=v5_eval_run(project,'ahr-v5-actual-gnu-whole-'+suffix,identity,players)
    for cutoff,label in ((2,'before'),(3,'after')):
        run='ahr-v5-actual-gnu-'+label+'-'+suffix
        partial,_=v5_eval_run(project,run,identity,v5_real_gnu_fixture_players(project),stop_after_events=cutoff)
        if partial['active'] is None: raise AssertionError('Expected active midgame fixture')
        resumed_digest=v5_real_gnu_fixture_subprocess(project,run,identity)
        if resumed_digest!=v5_digest(v5_eval_semantic(whole)):
            raise AssertionError('Fresh-process actual GNU chronology/RNG changed '+label+' GNU reply')
    lock_run=project/'checkpoints'/('ahr-v5-writer-fixture-'+suffix)
    with checkpoint_lock(lock_run):
        try:
            with checkpoint_lock(lock_run): pass
        except RuntimeError: pass
        else: raise AssertionError('Concurrent second writer was accepted')
    sandbox=project/'exports/validation'/('ahr-v5-lockbox-fixture-'+suffix)
    fixture_identity={'schema':'LOCKBOX_RECOVERY_FIXTURE_ONLY','candidate':'fixture-one'}
    v5_commit_final_lockbox(sandbox,'fixture-one',fixture_identity)
    v5_commit_final_lockbox(sandbox,'fixture-one',fixture_identity)
    try: v5_commit_final_lockbox(sandbox,'fixture-two',{'schema':'LOCKBOX_RECOVERY_FIXTURE_ONLY','candidate':'fixture-two'})
    except RuntimeError: pass
    else: raise AssertionError('Changed final lockbox accepted')
    orphan=project/'checkpoints'/('ahr-v5-orphan-fixture-'+suffix);orphan.mkdir(parents=True)
    atomic_write_bytes(orphan/'step-00000001.pt',b'ORPHAN_RECOVERY_FIXTURE')
    try: v5_eval_strict_load(orphan)
    except RuntimeError: pass
    else: raise AssertionError('Orphan payload without manifest silently restarted')
    # Pause before any game begins; repeated expired-deadline calls must not overwrite a committed step.
    pause_run='ahr-v5-deadline-pause-'+suffix
    paused,_=v5_eval_run(project,pause_run,identity,v5_real_gnu_fixture_players(project),deadline_unix=time.time()-1)
    paused_again,_=v5_eval_run(project,pause_run,identity,v5_real_gnu_fixture_players(project),deadline_unix=time.time()-1)
    if paused['results'] or paused['active'] is not None or paused['global_step']!=paused_again['global_step']:
        raise AssertionError('Wall deadline fabricated a result or changed a committed event')
    completed,_=v5_eval_run(project,pause_run,identity,v5_real_gnu_fixture_players(project))
    if v5_eval_semantic(completed)!=v5_eval_semantic(whole):raise AssertionError('Deadline pause/resume changed chronology')
    import shutil
    malformed=project/'checkpoints'/('ahr-v5-latest-pointer-fixture-'+suffix)
    shutil.copytree(project/'checkpoints'/pause_run,malformed)
    manifest=v5_read_json(malformed/'manifest.json');manifest['latest']='missing-committed-state.pt'
    v5_atomic_json(malformed/'manifest.json',manifest)
    try:v5_eval_strict_load(malformed)
    except RuntimeError:pass
    else:raise AssertionError('Malformed latest checkpoint pointer accepted')
    base['checks'].update(actual_gnu_before_reply_fresh_process=True,actual_gnu_after_reply_fresh_process=True,
        one_writer_rejection=True,single_final_lockbox_rejection=True,orphan_manifest_rejection=True,
        deadline_pause_exact_resume=True,malformed_latest_pointer_rejection=True)
    base['fixture_scope']='Exact CPU/RNG/stateful recovery wiring; actual pinned GNU calls before and after midgame resume. No playing-strength claim.'
    base['sources']=v5_source_identity(project)
    base['status']='PASS';v5_atomic_json(v5_report_path(project,'arena_validation.json'),base)
    v5_atomic_json(v5_report_path(project,'arena_recovery.json'),base)
    return base


## Competence and teacher gates

Engineering recovery and legality checks are separate from evidence that the model learned. The proposed observed engineering targets are 99% immediate capture and 95% for each selected bounded goal on reserved development families. These fractions are development thresholds; descriptive intervals are reported and no general optimality claim follows.

The actual one-second player and three-second offline teacher are measured on the same independent capture/survival tasks, selected by a predeclared unique urgent target and no competing own/enemy atari. The goals are balanced and families are distinct; insufficient rows fail measurement. An offline policy label requires a completed coherent comparison; the teacher must improve observed task success without regression, or both players must already exceed 95%. This selection avoids penalizing search for another immediate group obligation. All four goals remain in the complete tactical head holdout, including multi-threat positions. The search gate remains a bounded tactical diagnostic criterion.

If competence fails, a user-authorized 18-hour experiment can continue supervised diagnosis and remediation while withholding weak self-play labels. It must report FAIL, never relabel operational validation as playing strength. Final testing stays locked until actual development and fresh confirmation both reach at least 50% observed wins for the same candidate.

In [ ]:
@dataclass(frozen=True)
class V5GateConfig:
    immediate_capture_threshold: float = .99
    bounded_tactical_threshold: float = .95
    teacher_cases: int = 16
    teacher_minimum_success: float = .50
    teacher_minimum_gain: float = .05
    teacher_saturated_success: float = .95
    teacher_fast_s: float = 1.0
    teacher_offline_s: float = 3.0
    def __post_init__(self):
        for value in (self.immediate_capture_threshold,self.bounded_tactical_threshold,self.teacher_minimum_success,
                      self.teacher_minimum_gain,self.teacher_saturated_success):
            if not 0<=value<=1: raise ValueError('Gate fractions must lie in [0,1]')
        if type(self.teacher_cases) is not int or self.teacher_cases<4 or self.teacher_cases%2 or self.teacher_fast_s!=1.0 or self.teacher_offline_s<=self.teacher_fast_s:
            raise ValueError('Require at least four tasks and a slower offline teacher against one-second AHR')

def v5_wilson_interval(successes,count,z=1.959963984540054):
    if count<1: return None
    p=successes/count;denominator=1+z*z/count
    center=(p+z*z/(2*count))/denominator
    radius=z*math.sqrt(p*(1-p)/count+z*z/(4*count*count))/denominator
    return [max(0.,center-radius),min(1.,center+radius)]

def v5_gate_cases(project,tasks=None,count=16):
    if type(count) is not int or count<4 or count%2:raise ValueError('Require balanced even capture/survival task count')
    if tasks is None:tasks=v5_curriculum(project)['development']
    if not tasks:return []
    if any(row.get('split')!='development' for row in tasks):raise ValueError('Competence/search gates require reserved development tasks only')
    grouped={goal:[] for goal in ('capture_one_ply','survive_two_plies')};seen=set()
    for row in sorted(tasks,key=lambda r:r['family_id']):
        goal=row['tactical_goal']
        if goal not in grouped or row['family_id'] in seen:continue
        state=from_dict(row['state']);own=[];enemy=[]
        for stones,libs in groups_and_liberties(state.board,state.size):
            if len(libs)==1:
                (own if state.board[next(iter(stones))]==state.to_play else enemy).append(set(stones))
        target=set(row['tactical_targets'])
        capture=goal=='capture_one_ply' and len(enemy)==1 and not own and enemy[0]==target
        survival=goal=='survive_two_plies' and len(own)==1 and not enemy and own[0]==target
        if capture or survival:grouped[goal].append(row);seen.add(row['family_id'])
    # No outcome-based filtering: one unique urgent group and no competing atari,
    # deterministic family order, equal goal representation, fail insufficient rows.
    pairs=min(count//2,len(grouped['capture_one_ply']),len(grouped['survive_two_plies']))
    return [grouped[goal][i] for i in range(pairs) for goal in ('capture_one_ply','survive_two_plies')]

def v5_teacher_gate(project,model,tasks=None,device='cuda:1',config=None,force=False,teacher_search=None):
    project=Path(project).resolve();config=config if isinstance(config,V5GateConfig) else V5GateConfig(**(config or {}))
    cases=v5_gate_cases(project,tasks,config.teacher_cases)
    if model is None or len(cases)<config.teacher_cases:
        receipt={'status':'NOT_MEASURED','reason':'trained model and sufficient independent development capture/survival families required',
                 'cases_available':len(cases),'cases_required':config.teacher_cases,'task_selection':'balanced unique urgent target, no competing atari','full_game_optimality':'NOT_ESTABLISHED'}
        v5_atomic_json(v5_report_path(project,'search_teacher_gate.json'),receipt);return receipt
    fast=V5SearchConfig(time_budget_s=config.teacher_fast_s)
    offline=V5SearchConfig(**(teacher_search or {'time_budget_s':config.teacher_offline_s,'max_depth':4,'max_nodes':2048,'branch_width':12}))
    if offline.time_budget_s<=fast.time_budget_s:raise ValueError('Teacher gate requires a genuinely slower offline search')
    key={'sources':v5_source_identity(project),'model_weights_sha256':arena_tensor_digest(model.state_dict()),
         'config':asdict(config),'fast_search':asdict(fast),'offline_search':asdict(offline),
         'task_selection':'balanced unique urgent target, no competing atari; deterministic distinct development families','tasks_sha256':v5_digest([{k:r[k] for k in ('state','tactical_goal','tactical_targets','family_id')} for r in cases])}
    path=v5_report_path(project,'search_teacher_gate.json')
    if not force and path.exists():
        previous=v5_read_json(path)
        if previous.get('identity')==key:return previous
    previous_mode=model.training;model.eval();rows=[]
    try:
        with torch.inference_mode():
            v5_choose_move(model,initial_state(9,7.5),V5SearchConfig(time_budget_s=.2,max_depth=2,max_nodes=128))
        for case in cases:
            state=from_dict(case['state']);row={'family_id':case['family_id'],'goal':case['tactical_goal'],
                'scope':case['bounded_objective'],'target_stones':case['tactical_targets']}
            exact_case={'state':case['state'],'goal':case['tactical_goal'],'targets':case['tactical_targets']}
            for name,search in (('fast',fast),('offline',offline)):
                with torch.inference_mode(): result=v5_choose_move(model,state,search)
                action=result['action'];proof=v5_bounded_action(exact_case,action)
                depths=list(result.get('root_depths',{}).values())
                coherent=bool(result.get('comparison_complete') and result.get('policy_label_eligible')
                              and depths and len(set(depths))==1 and depths[0]>=2)
                row[name]={'action':action,'task_status':proof['status'],'success':proof['status']=='SAT',
                           'coherent_completed_comparison':coherent,'completed_depth':result.get('completed_depth'),
                           'root_candidates':len(depths),'elapsed_s':result.get('elapsed_s'),
                           'deadline_missed':result.get('deadline_missed',False),'reason':result.get('reason'),
                           'strategic_status':'UNKNOWN'}
            rows.append(row)
    finally:model.train(previous_mode)
    count=len(rows);fast_success=sum(r['fast']['success'] for r in rows);offline_success=sum(r['offline']['success'] for r in rows)
    fast_rate=fast_success/count;offline_rate=offline_success/count
    coherent=all(r['offline']['coherent_completed_comparison'] and not r['offline']['deadline_missed'] for r in rows)
    improved=offline_rate-fast_rate>=config.teacher_minimum_gain
    saturated=fast_rate>=config.teacher_saturated_success and offline_rate>=config.teacher_saturated_success
    passed=coherent and offline_rate>=config.teacher_minimum_success and offline_rate>=fast_rate and (improved or saturated)
    receipt={'status':'PASS' if passed else 'FAIL','identity':key,'count':count,
        'fast_success':fast_success,'offline_success':offline_success,'fast_success_rate':fast_rate,
        'offline_success_rate':offline_rate,'observed_gain':offline_rate-fast_rate,
        'fast_wilson_interval':v5_wilson_interval(fast_success,count),'offline_wilson_interval':v5_wilson_interval(offline_success,count),
        'checks':{'offline_coherent_completed_comparisons':coherent,'minimum_teacher_success':offline_rate>=config.teacher_minimum_success,
                  'teacher_nonregression':offline_rate>=fast_rate,'observed_improvement_or_saturation':improved or saturated},
        'rows':rows,'external_engine_labels':False,'final_data_used':False,
        'scope':'Actual whole-board AHR decisions on balanced unique urgent capture/survival targets with no competing atari, checked by independent bounded predicates. Full four-goal tactical holdout remains unchanged. This diagnostic gate is not a global best-move theorem.',
        'confidence_scope':'Intervals are descriptive for distinct selected development families; thresholds use observed fractions, not population guarantees',
        'full_game_optimality':'NOT_ESTABLISHED'}
    v5_atomic_json(path,receipt);return receipt

def v5_learning_gate(project,model,tasks=None,config=None,include_teacher=True,force=False):
    project=Path(project).resolve();config=config if isinstance(config,V5GateConfig) else V5GateConfig(**(config or {}))
    if tasks is None:
        try:tasks=v5_curriculum(project)['development']
        except FileNotFoundError:tasks=[]
    if model is None or not tasks:
        receipt={'status':'NOT_MEASURED','tactical_status':'NOT_MEASURED','teacher_status':'NOT_MEASURED',
            'reason':'Reserved development curriculum and actual model required','config':asdict(config),
            'full_game_strength':'NOT_ESTABLISHED'}
        v5_atomic_json(v5_report_path(project,'learning_gate.json'),receipt);return receipt
    if any(r.get('split')!='development' for r in tasks):raise ValueError('Learning gate must not use training/final rows')
    measured=v5_predict_tactical(model,tasks);by_goal=measured['by_goal']
    capture=by_goal.get('capture_one_ply',{}).get('top1_success')
    required_goals=set(V5_GOALS)
    available_goals=set(by_goal)
    capture_ok=capture is not None and capture>=config.immediate_capture_threshold
    bounded_ok=required_goals<=available_goals and all(by_goal[g]['top1_success']>=config.bounded_tactical_threshold for g in required_goals)
    teacher=v5_teacher_gate(project,model,tasks,config=config,force=force) if include_teacher else {'status':'NOT_MEASURED'}
    goals={}
    for goal,summary in by_goal.items():
        good=sum(r['correct'] for r in measured['rows'] if r['goal']==goal)
        goals[goal]={**summary,'successes':good,'wilson_interval':v5_wilson_interval(good,summary['count'])}
    receipt={'status':'PASS' if capture_ok and bounded_ok and teacher['status']=='PASS' else 'FAIL',
        'tactical_status':'PASS' if capture_ok and bounded_ok else 'FAIL','teacher_status':teacher['status'],
        'config':asdict(config),'source_identity':v5_source_identity(project),'model_weights_sha256':arena_tensor_digest(model.state_dict()),
        'checks':{'immediate_capture_threshold':capture_ok,'each_bounded_goal_threshold':bounded_ok,
                  'required_goals_present':required_goals<=available_goals,'teacher_improvement_gate':teacher['status']=='PASS'},
        'count':measured['count'],'top1_success':measured['top1_success'],'by_goal':goals,
        'teacher':{k:teacher.get(k) for k in ('status','count','fast_success_rate','offline_success_rate','observed_gain','checks')},
        'engineering_thresholds_are_observed':True,'full_game_strength':'NOT_ESTABLISHED',
        'scope':'Heldout goal/target-conditioned bounded action learning and measured offline AHR teacher quality, no final data'}
    v5_atomic_json(v5_report_path(project,'learning_gate.json'),receipt);return receipt

def v5_validate_gates(project,model=None,device='cuda:1',config=None,force=False):
    project=Path(project).resolve();sources=v5_source_identity(project)
    arena_path=v5_report_path(project,'arena_validation.json')
    arena=v5_read_json(arena_path) if arena_path.exists() else None
    if force or not arena or arena.get('sources')!=sources or arena.get('status')!='PASS':
        arena=v5_validate_arena_resume(project,device=device)
    search_path=v5_report_path(project,'model_search_validation.json')
    search=v5_read_json(search_path) if search_path.exists() else None
    search_sources=(search or {}).get('sources',(search or {}).get('source_identity'))
    if force or not search or search.get('status')!='PASS' or search_sources!=sources:
        search=v5_validate_model_search(project,device=device)
    checks={'arena_exact_resume':arena.get('status')=='PASS' and arena.get('exact_resume') is True,
            'actual_gnu_midgame_resume':arena['checks'].get('actual_gnu_before_reply_fresh_process',False) and arena['checks'].get('actual_gnu_after_reply_fresh_process',False),
            'one_writer_rejection':arena['checks'].get('one_writer_rejection',False),
            'lockbox_rejection':arena['checks'].get('single_final_lockbox_rejection',False),
            'unknown_is_unscored':arena['checks'].get('unknown_semantics',False),
            'native_model_and_coherent_search':search.get('status')=='PASS'}
    engineering='PASS' if all(checks.values()) else 'FAIL'
    try:learning=v5_learning_gate(project,model,config=config,force=force)
    except FileNotFoundError:learning={'status':'NOT_MEASURED','reason':'curriculum not generated'}
    receipt={'status':engineering,'engineering_status':engineering,'learning_status':learning['status'],
             'checks':checks,'sources':v5_source_identity(project),'learning':learning,
             'launch_policy':'Engineering PASS permits an explicitly bounded diagnostic learning run; gameplay requires measured learning PASS. Final remains locked until same frozen candidate reaches observed >=50% wins in development plus fresh confirmation.',
             'final_strength':'NOT_ESTABLISHED','created_unix':time.time()}
    v5_atomic_json(v5_report_path(project,'engineering_gates.json'),receipt);return receipt


In [ ]:
arena_validation=v5_validate_arena_resume(project,device='cuda:1')
from IPython.display import display, Markdown
import pandas as pd
display(Markdown('**Recovery: '+arena_validation['status']+'**. Exact fresh-process recovery passed, including real GNU replies. Playing strength has not been measured.'))
display(pd.DataFrame([{'Recovery check':name.replace('_',' '),'Result':'PASS' if value else 'FAIL'}
                      for name,value in arena_validation['checks'].items()]))


In [ ]:
# An isolated source-matched fixture proves that final access is rejected before any final openings or lockbox are created.
import shutil
sandbox=project/'exports/validation'/('ahr-v5-final-readiness-fixture-'+uuid.uuid4().hex[:10])
for relative in v5_source_identity(project):
    destination=sandbox/relative;destination.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(project/relative,destination)
fixture_candidate=sandbox/'checkpoints/explicit-readiness-fixture/frozen_candidate.bin'
atomic_write_bytes(fixture_candidate,b'EXPLICIT_READINESS_GUARD_FIXTURE_NOT_A_MODEL')
try:
    v5_require_final_ready(sandbox,'explicit-readiness-fixture',str(fixture_candidate.relative_to(sandbox)))
except RuntimeError as error:
    assert 'NOT_READY' in str(error)
else:
    raise AssertionError('Final guard allowed a candidate without development and confirmation evidence')
assert not (sandbox/'config/ahr_native_v5_final_lockbox.json').exists()
assert not (sandbox/'reports/ahr_native_v5/final').exists()
display(Markdown('**Final access guard: PASS.** A candidate without qualifying development and fresh confirmation was rejected. No final openings or final lockbox were created.'))


In [ ]:
# Control-flow fixtures only: actual05phase machine, mocked scientific workers, isolated checkpoint/report roots.
# No training, GNU strength game, final seed or service is launched by these checks.
from types import SimpleNamespace
from contextlib import redirect_stdout
import io
controller_rows=[]
for scenario in ('poor','promising','paused_development','unknown_parity','expired_budget','late_final','failed_confirmation_advances_training','failed_status_resume'):
    fixture_project=project/'exports/validation'/('ahr-v5-controller-flow-'+scenario+'-'+uuid.uuid4().hex[:8])
    private=dict(globals())
    v5_import_exports(project/'experiments/ahr_native_v5/05_budgeted_pipeline.ipynb',private)
    clock_value=[1700000000.0]
    private['time']=SimpleNamespace(time=lambda:clock_value[0],monotonic=lambda:clock_value[0])
    protocol={'schema':V5_SCHEMA,'training_run':'EXPLICIT_CONTROL_FLOW_FIXTURE',
              'training_targets':[16,32],'training':{'scheduled_games':32},'device':'cpu',
              'budget':{'requested_min_hours':12.,'learning_window_hours':16.,'target_hours':18.,'hard_limit_hours':24.,'segment_seconds':1800.},
              'development_games':16,'final_development_games':32,'confirmation_games':32,'final_games':200,
              'stop_after_unproductive_reviews':2,'off_instance_backup':'FIXTURE_ONLY','configuration_sha256':'FIXTURE_ONLY'}
    calls=[];eval_counts=Counter();checkpoint_by_target={};commit_statuses=[];resumed_clock_contract=None
    private['v5_protocol_read']=lambda p:copy.deepcopy(protocol)
    original_commit=private['v5_controller_commit']
    def record_commit(p,state,*args,**kwargs):
        commit_statuses.append(state['metrics']['status'])
        return original_commit(p,state,*args,**kwargs)
    private['v5_controller_commit']=record_commit
    def learning(p,run,cfg,run_until_games=None,deadline_unix=None):
        calls.append(('learning',run_until_games));clock_value[0]+=1
        checkpoint_by_target[run_until_games]='EXPLICIT_FIXTURE_CHECKPOINT_G'+str(run_until_games)
        return {'status':'RUNNING','games_finished':run_until_games,'optimizer_steps':1,'phase':'FIXTURE_ONLY'}
    def evaluate(p,training,run,split='development',games=16,checkpoint_path=None,device='cpu',deadline_unix=None):
        candidate_path=checkpoint_path or checkpoint_by_target[max(checkpoint_by_target)]
        calls.append((split,games,run,candidate_path));eval_counts[(split,run)]+=1;clock_value[0]+=1
        if len(calls)>30:raise AssertionError('Controller stopped making progress in fixture '+scenario)
        wins=0
        if split=='development' and games==32 and scenario in ('promising','paused_development','unknown_parity','late_final'):wins=games//2
        if scenario=='failed_confirmation_advances_training' and split=='development':wins=games//2
        if split=='confirmation':
            wins=0 if scenario=='failed_confirmation_advances_training' and sum(eval_counts[k] for k in eval_counts if k[0]=='confirmation')==1 else games//2
        if split=='final':wins=120
        unknown=1 if scenario=='unknown_parity' and split=='development' and games==32 else 0
        status='IN_PROGRESS' if scenario=='paused_development' and split=='development' and eval_counts[(split,run)]==1 else 'COMPLETE'
        if scenario=='late_final' and split=='final':
            status='IN_PROGRESS';clock_value[0]=1700000000.0+24*3600
        return {'status':status,'split':split,'eval_run':run,'training_run':training,'games_planned':games,
                'games_complete':games-unknown if status=='COMPLETE' else 0,'wins':wins,'unknown':unknown,
                'pending':0 if status=='COMPLETE' else games,'wins_over_planned':wins/games,
                'observed_half_wins':status=='COMPLETE' and unknown==0 and wins>=games/2,
                'mean_score_margin':-70 if not wins else 5,'checkpoint_path':candidate_path,
                'checkpoint_sha256':'EXPLICIT_FIXTURE_SHA_'+candidate_path,'optimizer_steps':1,'report_path':'EXPLICIT_FIXTURE_REPORT'}
    private['V5TrainConfig']=lambda **kwargs:kwargs
    private['v5_run_learning']=learning;private['v5_run_gnu_evaluation']=evaluate
    private['v5_receipt_wipeout']=lambda p,r:1.0 if not r['wins'] else 0.0
    private['v5_local_bundle']=lambda p,state:{'status':'EXPLICIT_MOCK_BUNDLE','not_a_backup':True}
    with redirect_stdout(io.StringIO()):
        if scenario=='expired_budget':
            state=private['v5_controller_initial'](protocol)
            with checkpoint_lock(fixture_project/'checkpoints'/private['V5_CONTROLLER']):
                private['v5_controller_commit'](fixture_project,state,best=True)
            clock_value[0]+=25*3600
        if scenario=='failed_status_resume':
            state=private['v5_controller_initial'](protocol)
            state['metrics']['status']='INTERRUPTED_OR_FAILED'
            state['metrics']['last_error']='EXPLICIT_CONTROL_FLOW_FIXTURE_SIMULATED_FAILURE'
            resumed_clock_contract={k:state['metrics'][k] for k in ('started_unix','learning_deadline_unix','target_deadline_unix','hard_deadline_unix')}
            with checkpoint_lock(fixture_project/'checkpoints'/private['V5_CONTROLLER']):
                private['v5_controller_commit'](fixture_project,state,best=True)
            clock_value[0]+=900
        result=private['v5_run_budgeted_pipeline'](fixture_project)
        started=result['started_unix'];first_calls=len(calls)
        again=private['v5_run_budgeted_pipeline'](fixture_project)
    assert again['started_unix']==started and len(calls)==first_calls
    strength_calls=[c for c in calls if c[0] in ('development','confirmation','final')]
    if scenario in ('poor','unknown_parity'):
        assert result['status']=='BUDGET_COMPLETE_FINAL_NOT_READY'
        assert not any(c[0] in ('confirmation','final') for c in strength_calls)
        assert [c[1] for c in strength_calls]==[16,32]
    elif scenario in ('promising','paused_development'):
        assert result['status']=='TARGET_OBSERVED'
        assert [c[0] for c in strength_calls if c[0]!='development']==['confirmation','final']
        assert next(c[1] for c in strength_calls if c[0]=='final')==200
        if scenario=='paused_development':
            for run in {c[2] for c in strength_calls if c[0]=='development'}:assert eval_counts[('development',run)]==2
    elif scenario=='expired_budget':
        assert result['status']=='HARD_BUDGET_REACHED' and not calls
    elif scenario=='late_final':
        assert result['status']=='HARD_BUDGET_REACHED_FINAL_INCOMPLETE'
        completion=v5_read_json(fixture_project/'reports/ahr_native_v5/completion.json')
        assert completion['final']['status']=='IN_PROGRESS'
    elif scenario=='failed_confirmation_advances_training':
        assert result['status']=='TARGET_OBSERVED'
        assert [c[1] for c in calls if c[0]=='learning']==[16,32]
        development_calls=[c for c in strength_calls if c[0]=='development']
        assert len(development_calls)==2 and development_calls[0][3]!=development_calls[1][3]
        assert [c[0] for c in strength_calls]==['development','confirmation','development','confirmation','final']
    elif scenario=='failed_status_resume':
        assert result['status']=='BUDGET_COMPLETE_FINAL_NOT_READY'
        assert 'RUNNING' in commit_statuses[1:]
        assert 'INTERRUPTED_OR_FAILED' not in commit_statuses[1:]
        assert result['last_error'] is None
        assert all(result[k]==value for k,value in resumed_clock_contract.items())
        assert [c[1] for c in calls if c[0]=='learning']==[16,32]
    controller_rows.append({'Control flow fixture':scenario.replace('_',' '),'Expected status':result['status'],'Result':'PASS'})
display(Markdown('**Controller phase checks: PASS.** These eight isolated mock-worker tests check request sizes, UNKNOWN readiness, paused requests and fixed budgets. They provide no model or playing-strength evidence.'))
display(pd.DataFrame(controller_rows))


In [ ]:
# Optional actual GPU teacher diagnostic. Explicit flag avoids duplicating GPU work in default recovery Run All.
if os.environ.get('AHR_NATIVE_V5_TEACHER_DIAGNOSTIC')=='1':
    generation_dir=project/'checkpoints/ahr-v5-curriculum-generation'
    generation_manifest=v5_read_json(generation_dir/'manifest.json')
    latest=next(e for e in generation_manifest['entries'] if e['file']==generation_manifest['latest'])
    generated=_checkpoint_entry_state(generation_dir,latest)
    assert generated is not None
    development=[r for r in generated['examples'] if r['split']=='development']
    unique_urgent=[]
    for row in development:
        if row['tactical_goal'] not in ('capture_one_ply','survive_two_plies'):continue
        state=from_dict(row['state'])
        own=[];opponent=[]
        for stones,libs in groups_and_liberties(state.board,state.size):
            if len(libs)==1:
                (own if state.board[next(iter(stones))]==state.to_play else opponent).append(set(stones))
        target=set(row['tactical_targets'])
        qualifies=(row['tactical_goal']=='capture_one_ply' and len(opponent)==1 and not own and opponent[0]==target)
        qualifies=qualifies or (row['tactical_goal']=='survive_two_plies' and len(own)==1 and not opponent and own[0]==target)
        if qualifies:unique_urgent.append(row)
    initial_model,initial_provenance=v5_model_initialization(project,device='cuda:1')
    # Preserve the earlier raw/multi-threat and unique-target measurements; they motivated the predeclared selection.
    measured=v5_teacher_gate(project,initial_model,tasks=development,teacher_search={'time_budget_s':3.0,'max_depth':4,'max_nodes':2048,'branch_width':12},force=True)
    chosen_cases=v5_gate_cases(project,development,16)
    v5_atomic_json(v5_report_path(project,'teacher_baseline_after_selection_tasks.json'),{'selected_tasks':chosen_cases})
    diagnostic={'schema':'v5_teacher_baseline_after_predeclared_selection','source_identity':v5_source_identity(project),
                'generation_checkpoint':latest,'development_available':len(development),
                'unique_urgent_available':dict(Counter(r['tactical_goal'] for r in unique_urgent)),
                'initial_model_provenance':initial_provenance,'selected_target_gate':measured,
                'prior_diagnostic_preserved':'reports/ahr_native_v5/teacher_baseline_diagnostic.json',
                'scope':'Actual initialized AHR decisions with final matching search configuration and predeclared balanced unique urgent targets; diagnostic only, no training or final games'}
    v5_atomic_json(v5_report_path(project,'teacher_baseline_after_selection.json'),diagnostic)
    summary=[{'Actual baseline task selection':'predeclared balanced unique urgent targets','Gate':measured['status'],
              'Fast success':measured.get('fast_success_rate'),'Offline success':measured.get('offline_success_rate'),'Observed gain':measured.get('observed_gain')}]
    display(Markdown('**Actual GPU teacher diagnostic.** These measurements concern selected bounded tactical targets on partial development data, and do not establish full-game strength.'))
    display(pd.DataFrame(summary))
else:
    saved_diagnostic=v5_report_path(project,'teacher_baseline_after_selection.json')
    if saved_diagnostic.exists():
        recorded=v5_read_json(saved_diagnostic)['selected_target_gate']
        display(Markdown('**Saved actual GPU teacher diagnostic.** This recorded selected-target measurement is shown without rerunning GPU search. It does not establish full-game strength. Set `AHR_NATIVE_V5_TEACHER_DIAGNOSTIC=1` to reproduce it.'))
        display(pd.DataFrame([{'Recorded gate':recorded['status'],'Fast success':recorded.get('fast_success_rate'),
                              'Offline success':recorded.get('offline_success_rate'),'Observed gain':recorded.get('observed_gain')}]))
    else:
        print('Teacher diagnostic not measured. Set AHR_NATIVE_V5_TEACHER_DIAGNOSTIC=1 after generating development curriculum.')


## Run a development block or resume a protected arena

The controller imports these definitions and starts strength matches with a trained checkpoint. A manual example is `v5_run_gnu_evaluation(project, training_run, eval_run, split='development', games=16, checkpoint_path=protected_path, device='cuda:1')`. Resume by repeating precisely that call. Fresh confirmation uses `split='confirmation'` and 32 or 64 games. Final requires `split='final', games=200` and the same protected candidate after readiness passes.

Do not remove a damaged manifest or change a run ID to conceal a failed resume. Inspect preserved artifacts, repair the underlying issue, and record an explicit new experiment identity if the source or protocol changes. This notebook's Run All performs recovery fixtures only; it does not consume final openings or start a long run.